# Amazon 2018 Industrial 数据预处理

本 notebook 通过 `data/amazon18_data_process.py` 执行标准 MiniOneRec 预处理。先运行前两个单元检查路径，再按需运行最后一个单元。

In [14]:
# 配置当前仓库、原始数据和输出目录。
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path('D:/转码ing/MiniOneRec-main')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'Amazon18'
# 使用新目录保存 2016-10 设置的结果，避免覆盖此前自动扩展到 2013-10 的结果。
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'Amazon18_2016_10_2018_11'
DATASET = 'Industrial_and_Scientific'

REVIEWS_FILE = RAW_DIR / 'Industrial_and_Scientific.json'
METADATA_FILE = RAW_DIR / 'meta_Industrial_and_Scientific.json'
PREPROCESS_SCRIPT = PROJECT_ROOT / 'data' / 'amazon18_data_process.py'

USER_K = 5  # 每位用户至少保留 5 条有效交互记录。
ITEM_K = 5  # 当前官方脚本未实际使用；商品也会按 USER_K=5 筛选。
START_YEAR, START_MONTH = 2016, 10
END_YEAR, END_MONTH = 2018, 11

In [15]:
# 运行前只检查文件路径和输出目录，不读取完整数据，也不创建结果。
required_paths = [PREPROCESS_SCRIPT, REVIEWS_FILE, METADATA_FILE]
missing_paths = [path for path in required_paths if not path.is_file()]

if missing_paths:
    raise FileNotFoundError('缺少以下文件：\n' + '\n'.join(str(path) for path in missing_paths))

target_dir = OUTPUT_DIR / DATASET
if target_dir.exists():
    raise FileExistsError(
        f'输出目录已存在：{target_dir}\n'
        '为避免覆盖已有结果，请修改 OUTPUT_DIR 后再运行。'
    )

print('输入检查通过。')
print(f'Reviews:  {REVIEWS_FILE} ({REVIEWS_FILE.stat().st_size / 1024 / 1024:.2f} MiB)')
print(f'Metadata: {METADATA_FILE} ({METADATA_FILE.stat().st_size / 1024 / 1024:.2f} MiB)')
print(f'输出目录: {target_dir}')
print('提示：当前官方脚本实际只使用 USER_K；ITEM_K 参数在脚本中尚未生效。')

输入检查通过。
Reviews:  D:\转码ing\MiniOneRec-main\data\raw\Amazon18\Industrial_and_Scientific.json (740.17 MiB)
Metadata: D:\转码ing\MiniOneRec-main\data\raw\Amazon18\meta_Industrial_and_Scientific.json (607.84 MiB)
输出目录: D:\转码ing\MiniOneRec-main\data\Amazon18_2016_10_2018_11\Industrial_and_Scientific
提示：当前官方脚本实际只使用 USER_K；ITEM_K 参数在脚本中尚未生效。


In [16]:
# 确认上一个单元通过后，再运行此单元开始完整预处理。
# subprocess 会实时显示官方脚本的过滤和切分日志。
command = [
    sys.executable, str(PREPROCESS_SCRIPT),
    '--dataset', DATASET,
    '--reviews_file', str(REVIEWS_FILE),
    '--metadata_file', str(METADATA_FILE),
    '--user_k', str(USER_K),
    '--item_k', str(ITEM_K),
    '--st_year', str(START_YEAR),
    '--st_month', str(START_MONTH),
    '--ed_year', str(END_YEAR),
    '--ed_month', str(END_MONTH),
    '--output_path', str(OUTPUT_DIR),
]

print('将执行命令：')
print(' '.join(command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

将执行命令：
c:\Users\k\.conda\envs\tf_env\python.exe D:\转码ing\MiniOneRec-main\data\amazon18_data_process.py --dataset Industrial_and_Scientific --reviews_file D:\转码ing\MiniOneRec-main\data\raw\Amazon18\Industrial_and_Scientific.json --metadata_file D:\转码ing\MiniOneRec-main\data\raw\Amazon18\meta_Industrial_and_Scientific.json --user_k 5 --item_k 5 --st_year 2016 --st_month 10 --ed_year 2018 --ed_month 11 --output_path D:\转码ing\MiniOneRec-main\data\Amazon18_2016_10_2018_11


CompletedProcess(args=['c:\\Users\\k\\.conda\\envs\\tf_env\\python.exe', 'D:\\转码ing\\MiniOneRec-main\\data\\amazon18_data_process.py', '--dataset', 'Industrial_and_Scientific', '--reviews_file', 'D:\\转码ing\\MiniOneRec-main\\data\\raw\\Amazon18\\Industrial_and_Scientific.json', '--metadata_file', 'D:\\转码ing\\MiniOneRec-main\\data\\raw\\Amazon18\\meta_Industrial_and_Scientific.json', '--user_k', '5', '--item_k', '5', '--st_year', '2016', '--st_month', '10', '--ed_year', '2018', '--ed_month', '11', '--output_path', 'D:\\转码ing\\MiniOneRec-main\\data\\Amazon18_2016_10_2018_11'], returncode=0)

运行完成后，预期结果位于 `data/Amazon18/Industrial_and_Scientific/`：

- `Industrial_and_Scientific.train.inter`、`valid.inter`、`test.inter`：按时间切分的原始 item-id 序列，下一步会被转换为 SID 序列。
- `Industrial_and_Scientific.item.json`：保留商品的 title、description、brand、categories；下一步文本 embedding 直接读取它。
- `Industrial_and_Scientific.review.json`：保留交互对应的评论和摘要。
- `Industrial_and_Scientific.inter.json`：用户重编号后的完整物品交互列表。
- `Industrial_and_Scientific.user2id`、`item2id`：原始 ID 到内部整数 ID 的映射。

后续顺序：`item.json` → 文本编码 → `.emb-qwen-td.npy` → SID 构建 → SID 化 train/valid/test 数据。

In [17]:
# 读取已生成的中间文件，只打印基本统计和少量样本。
# 这些操作只读取输出文件，不会修改已有预处理结果。

from pathlib import Path
import json

target_dir = OUTPUT_DIR / DATASET

train_file = target_dir / f"{DATASET}.train.inter"
valid_file = target_dir / f"{DATASET}.valid.inter"
test_file = target_dir / f"{DATASET}.test.inter"
item_file = target_dir / f"{DATASET}.item.json"
review_file = target_dir / f"{DATASET}.review.json"
inter_file = target_dir / f"{DATASET}.inter.json"
user_map_file = target_dir / f"{DATASET}.user2id"
item_map_file = target_dir / f"{DATASET}.item2id"

required_files = [
    train_file,
    valid_file,
    test_file,
    item_file,
    review_file,
    inter_file,
    user_map_file,
    item_map_file,
]

missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "缺少以下预处理输出文件：\n"
        + "\n".join(str(path) for path in missing_files)
    )

print("=== 输出目录 ===")
print(target_dir)

print("\n=== 文件大小 ===")
for path in required_files:
    print(f"{path.name}: {path.stat().st_size / 1024 / 1024:.2f} MiB")

print("\n=== train / valid / test 序列样本数 ===")
for split_name, path in [
    ("train", train_file),
    ("valid", valid_file),
    ("test", test_file),
]:
    with path.open("r", encoding="utf-8") as file:
        line_count = sum(1 for _ in file)

    print(f"{split_name}: {line_count - 1:,} samples")

print("\n=== train.inter 前 3 行 ===")
with train_file.open("r", encoding="utf-8") as file:
    for _ in range(4):
        print(file.readline().rstrip())

with item_file.open("r", encoding="utf-8") as file:
    item_features = json.load(file)

with review_file.open("r", encoding="utf-8") as file:
    review_data = json.load(file)

with inter_file.open("r", encoding="utf-8") as file:
    user_item_sequences = json.load(file)

print("\n=== JSON 条目数 ===")
print(f"item.json 商品数: {len(item_features):,}")
print(f"review.json 评论条目数: {len(review_data):,}")
print(f"inter.json 用户数: {len(user_item_sequences):,}")

print("\n=== item.json 前 3 个商品 ===")
for item_id, features in list(item_features.items())[:3]:
    print(
        {
            "item_id": item_id,
            "title": features.get("title", ""),
            "description_preview": features.get("description", "")[:160],
            "brand": features.get("brand", ""),
        }
    )

print("\n=== review.json 前 3 条评论 ===")
for key, value in list(review_data.items())[:3]:
    print(
        {
            "key": key,
            "summary": value.get("summary", ""),
            "review_preview": value.get("review", "")[:160],
        }
    )

print("\n=== inter.json 第一个用户序列 ===")
first_user_id, first_item_sequence = next(iter(user_item_sequences.items()))
print(
    {
        "user_id": first_user_id,
        "sequence_length": len(first_item_sequence),
        "item_id_sequence_preview": first_item_sequence[:20],
    }
)

with user_map_file.open("r", encoding="utf-8") as file:
    user_map_count = sum(1 for _ in file)

with item_map_file.open("r", encoding="utf-8") as file:
    item_map_count = sum(1 for _ in file)

print("\n=== ID 映射数量 ===")
print(f"user2id 映射数: {user_map_count:,}")
print(f"item2id 映射数: {item_map_count:,}")

=== 输出目录 ===
D:\转码ing\MiniOneRec-main\data\Amazon18_2016_10_2018_11\Industrial_and_Scientific

=== 文件大小 ===
Industrial_and_Scientific.train.inter: 0.75 MiB
Industrial_and_Scientific.valid.inter: 0.11 MiB
Industrial_and_Scientific.test.inter: 0.11 MiB
Industrial_and_Scientific.item.json: 2.65 MiB
Industrial_and_Scientific.review.json: 11.71 MiB
Industrial_and_Scientific.inter.json: 0.54 MiB
Industrial_and_Scientific.user2id: 0.12 MiB
Industrial_and_Scientific.item2id: 0.05 MiB

=== train / valid / test 序列样本数 ===
train: 29,458 samples
valid: 3,682 samples
test: 3,683 samples

=== train.inter 前 3 行 ===
user_id:token	item_id_list:token_seq	item_id:token
3545	245	245
4150	939	939
3172	1338	1338

=== JSON 条目数 ===
item.json 商品数: 3,106
review.json 评论条目数: 40,468
inter.json 用户数: 6,300

=== item.json 前 3 个商品 ===
{'item_id': '0', 'title': 'SUPCO SPP6 Relay/Capacitor Hard Start Kit with 500% Increase Starting Torque', 'description_preview': "['Relay/CAPACITOR hard start kit 500% incr starting torqu

In [ ]:
# 重复样本占比：用户连续两次交互同一个商品的比例

from pathlib import Path

target_dir = Path(
    "D:/转码ing/MiniOneRec-main/"
    "data/Amazon18_2016_10_2018_11/Industrial_and_Scientific"
)

for split_name in ["train", "valid", "test"]:
    split_file = target_dir / f"Industrial_and_Scientific.{split_name}.inter"

    total_count = 0
    repeat_target_count = 0
    empty_history_count = 0

    with split_file.open("r", encoding="utf-8") as file:
        next(file)

        for line in file:
            user_id, history_text, target_item_id = line.rstrip().split("\t")
            history_item_ids = history_text.split()

            total_count += 1

            if not history_item_ids:
                empty_history_count += 1
            elif history_item_ids[-1] == target_item_id:
                repeat_target_count += 1

    print(f"\n=== {split_name} ===")
    print(f"总样本数: {total_count:,}")
    print(f"空历史样本数: {empty_history_count:,}")
    print(f"目标等于最后一个历史商品: {repeat_target_count:,}")
    print(f"重复目标占比: {repeat_target_count / total_count:.2%}")


=== train ===
总样本数: 29,458
空历史样本数: 0
目标等于最后一个历史商品: 2,080
重复目标占比: 7.06%

=== valid ===
总样本数: 3,682
空历史样本数: 0
目标等于最后一个历史商品: 181
重复目标占比: 4.92%

=== test ===
总样本数: 3,683
空历史样本数: 0
目标等于最后一个历史商品: 103
重复目标占比: 2.80%


# 中间结果观察

## 1. 读取原始 reviews 与 metadata

In [1]:
# 这段代码会重新读取完整原始数据并执行 k-core，因此需要一些时间和内存。
# 但只展示少量中间结果，不会改动 data/Amazon18 中已有的预处理结果。

from pathlib import Path
from datetime import datetime, timezone
from types import SimpleNamespace
from tempfile import TemporaryDirectory
import sys

PROJECT_ROOT = Path("D:/转码ing/MiniOneRec-main")
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "Amazon18"

DATASET = "Industrial_and_Scientific"
REVIEWS_FILE = RAW_DIR / "Industrial_and_Scientific.json"
METADATA_FILE = RAW_DIR / "meta_Industrial_and_Scientific.json"

USER_K = 5
START_YEAR, START_MONTH = 2016, 10
END_YEAR, END_MONTH = 2018, 11
SAMPLE_SIZE = 3

# 导入仓库中的官方预处理模块。
sys.path.insert(0, str(PROJECT_ROOT / "data"))
import amazon18_data_process as prep

start_timestamp = prep.get_timestamp_start(START_YEAR, START_MONTH)
end_timestamp = prep.get_timestamp_start(END_YEAR, END_MONTH)

reviews = prep.load_reviews_json2csv_style(
    DATASET,
    reviews_file=str(REVIEWS_FILE),
)

metadata, id_title, removed_items = prep.load_metadata_json2csv_style(
    DATASET,
    metadata_file=str(METADATA_FILE),
)

print("=== 原始输入 ===")
print(f"reviews 数量: {len(reviews):,}")
print(f"metadata 数量: {len(metadata):,}")
print(f"有合格标题的商品数: {len(id_title):,}")
print(f"因标题不合规而标记删除的商品数: {len(removed_items):,}")

Processing metadata: 100%|██████████| 167442/167442 [00:00<00:00, 243267.19it/s]

=== 原始输入 ===
reviews 数量: 1,758,333
metadata 数量: 167,442
有合格标题的商品数: 147,406
因标题不合规而标记删除的商品数: 18,281


## 2. k-core filtering

In [2]:
from types import SimpleNamespace

ITEM_K = 5

args = SimpleNamespace(
    dataset=DATASET,
    metadata_file=str(METADATA_FILE),
    user_k=USER_K,
    item_k=ITEM_K,
    st_year=START_YEAR,
    st_month=START_MONTH,
    ed_year=END_YEAR,
    ed_month=END_MONTH,
)

start_timestamp = prep.get_timestamp_start(START_YEAR, START_MONTH)
end_timestamp = prep.get_timestamp_start(END_YEAR, END_MONTH)

# 该函数会在商品数不足 3000 时自动向前扩展起始年份。
result = prep.process_dataset_recursive(
    args=args,
    metadata=None,
    reviews=reviews,
    start_timestamp=start_timestamp,
    end_timestamp=end_timestamp,
)

filtered_reviews, user_counts, item_counts, metadata, id_title = result

print("=== process_dataset_recursive 最终输出 ===")
print(f"最终起始年份: {args.st_year}-{args.st_month}")
print(f"保留 reviews 数: {len(filtered_reviews):,}")
print(f"保留用户数: {len(user_counts):,}")
print(f"保留商品数: {len(item_counts):,}")

Processing metadata: 100%|██████████| 167442/167442 [00:00<00:00, 260839.45it/s]


Loaded 167442 metadata items, 147406 with valid titles
Performing k-core filtering...


K-core filtering: 100%|██████████| 1758333/1758333 [00:02<00:00, 822958.52it/s]


Users: 464396, Items: 87595, Reviews: 595544, Density: 1.4640168196914516e-05


K-core filtering: 100%|██████████| 595544/595544 [00:00<00:00, 859146.41it/s]


...

Users: 6303, Items: 3108, Reviews: 43143, Density: 0.002202328118558485


K-core filtering: 100%|██████████| 43143/43143 [00:00<00:00, 359108.67it/s]


Users: 6303, Items: 3106, Reviews: 43135, Density: 0.0022033375903439925


K-core filtering: 100%|██████████| 43135/43135 [00:00<00:00, 367693.34it/s]


Users: 6300, Items: 3106, Reviews: 43123, Density: 0.002203773546336328
After filtering: 6300 users, 3106 items, 43123 reviews
=== process_dataset_recursive 最终输出 ===
最终起始年份: 2013-10
保留 reviews 数: 43,123
保留用户数: 6,300
保留商品数: 3,106


In [ ]:
# result = (
#     filtered_reviews,  # 43,123 条 review 的完整列表
#     user_counts,       # 6,300 个用户的完整字典
#     item_counts,       # 3,106 个商品的完整字典
#     metadata,          # 167,442 条 metadata 的完整列表
#     id_title,          # 147,406 个标题映射
# )

In [6]:
filtered_reviews, user_counts, item_counts, metadata, id_title = result

print("=== 前 3 条 k-core 后的 review ===")
for record in filtered_reviews[:SAMPLE_SIZE]:
    print(
        {
            "reviewerID": record["reviewerID"],
            "asin": record["asin"],
            "overall": record["overall"],
            "unixReviewTime": record["unixReviewTime"],
            "summary": record.get("summary", ""),
            "reviewText_preview": record.get("reviewText", "")[:120],
        }
    )

=== 前 3 条 k-core 后的 review ===
{'reviewerID': 'A1JB7HFWHRYHT7', 'asin': 'B0000223SI', 'overall': 5.0, 'unixReviewTime': 1511740800, 'summary': "Couldn't have been happier with it's performance", 'reviewText_preview': 'This worked really well for what I used it for. So for my purposes it is getting full marks. This is an all around great'}
{'reviewerID': 'AUL5LCV4TT73P', 'asin': 'B0000223SK', 'overall': 4.0, 'unixReviewTime': 1515801600, 'summary': 'As advertised', 'reviewText_preview': 'As advertised'}
{'reviewerID': 'A1V3I3L5JKO7TM', 'asin': 'B0000223SK', 'overall': 5.0, 'unixReviewTime': 1507334400, 'summary': 'seems like a pretty good value as opposed to buying it ...', 'reviewText_preview': 'seems like a pretty good value as opposed to buying it at the big box stores by the sheet.'}


In [7]:
print("=== 前 3 个用户及其交互数 ===")
print(list(user_counts.items())[:SAMPLE_SIZE])

print("=== 前 3 个商品及其交互数 ===")
print(list(item_counts.items())[:SAMPLE_SIZE])

=== 前 3 个用户及其交互数 ===
[('A1JB7HFWHRYHT7', 8), ('AUL5LCV4TT73P', 8), ('A1V3I3L5JKO7TM', 5)]
=== 前 3 个商品及其交互数 ===
[('B0000223SI', 6), ('B0000223SK', 16), ('B0000223UV', 20)]


In [8]:
print("=== 前 3 条 metadata ===")
for record in metadata[:SAMPLE_SIZE]:
    print(
        {
            "asin": record.get("asin"),
            "title": record.get("title"),
            "description_preview": str(record.get("description", ""))[:120],
        }
    )

print("=== 前 3 个 ASIN → title 映射 ===")
print(list(id_title.items())[:SAMPLE_SIZE])

=== 前 3 条 metadata ===
{'asin': '0176496920', 'title': 'Turning Technologies Response Card (RCRF-03)', 'description_preview': "['RCRF-03 - Works on all Turning Technologies RF systems along with RCRF-01 and RCRF-02.']"}
{'asin': '0692782109', 'title': 'R-Cat 692782109 EKG Badge with Arrhythmia Pocket Booklet', 'description_preview': "['The only laminated pocket tool that provides 41 six-second keg rhythm strips now all calibrated to the exact size of a"}
{'asin': '0781776848', 'title': "Anatomical Chart Company's Illustrated Pocket Anatomy: The Spinal Nerves & the Autonomic Nervous System Study Guide", 'description_preview': '[\'<div>\', "Now in its Second Edition,<B>The Spinal Nerves and Autonomic Nervous System Illustrated&#160;Pocket Anatomy <'}
=== 前 3 个 ASIN → title 映射 ===
[('0176496920', 'Turning Technologies Response Card (RCRF-03)'), ('0692782109', 'R-Cat 692782109 EKG Badge with Arrhythmia Pocket Booklet'), ('0781776848', "Anatomical Chart Company's Illustrated Pocket Anatomy:

## 3. 用户与商品重编号

In [9]:
user2items, user2index, item2index, interactions = (
    prep.convert_inters2dict_amazon18_style(filtered_reviews)
)

print("\n=== convert_inters2dict_amazon18_style 输出 ===")
print(f"user2items 用户数: {len(user2items):,}")
print(f"user2index 用户数: {len(user2index):,}")
print(f"item2index 商品数: {len(item2index):,}")
print(f"interactions 数: {len(interactions):,}")
print("前 3 个 user2index 映射:", list(user2index.items())[:SAMPLE_SIZE])
print("前 3 个 item2index 映射:", list(item2index.items())[:SAMPLE_SIZE])
print("第一个用户的 item-id 序列:", next(iter(user2items.items())))


=== convert_inters2dict_amazon18_style 输出 ===
user2items 用户数: 6,300
user2index 用户数: 6,300
item2index 商品数: 3,106
interactions 数: 43,123
前 3 个 user2index 映射: [('A1JB7HFWHRYHT7', 0), ('AUL5LCV4TT73P', 1), ('A1V3I3L5JKO7TM', 2)]
前 3 个 item2index 映射: [('B0002YTLFE', 0), ('B000BQKBCK', 1), ('B0002QZ4XK', 2)]
第一个用户的 item-id 序列: (0, [0, 1, 2, 3, 4, 5, 6, 7])


## 4. 构造“历史序列 → 下一个商品”样本

In [10]:
interaction_list = prep.generate_interaction_list_json2csv_style(
    filtered_reviews,
    user2index,
    item2index,
    id_title,
)

print("\n=== generate_interaction_list_json2csv_style 输出 ===")
print(f"序列样本数: {len(interaction_list):,}")

for index, sample in enumerate(interaction_list[:SAMPLE_SIZE], start=1):
    print(f"\n样本 {index}")
    print("原始用户 ID:", sample[0])
    print("历史原始 ASIN:", sample[1])
    print("目标原始 ASIN:", sample[2])
    print("历史内部 item-id:", sample[3])
    print("目标内部 item-id:", sample[4])
    print("历史标题:", sample[5])
    print("目标标题:", sample[6])
    print(
        "目标时间:",
        datetime.fromtimestamp(int(sample[10]), tz=timezone.utc).isoformat(),
    )

Creating interaction sequences: 100%|██████████| 6300/6300 [00:00<00:00, 20332.58it/s]


=== generate_interaction_list_json2csv_style 输出 ===
序列样本数: 36,823

样本 1
原始用户 ID: A2WJIDZ6PKRE8R
历史原始 ASIN: ['B0001MSC84']
目标原始 ASIN: B0001MSC84
历史内部 item-id: [245]
目标内部 item-id: 245
历史标题: ['Rubbermaid Commercial BRUTE Heavy-Duty Round Waste/Utility Container with Venting Channels, 20-gallon, Gray (FG262000GRAY)']
目标标题: Rubbermaid Commercial BRUTE Heavy-Duty Round Waste/Utility Container with Venting Channels, 20-gallon, Gray (FG262000GRAY)
目标时间: 2013-10-01T00:00:00+00:00

样本 2
原始用户 ID: A1VNL7MD93UMB
历史原始 ASIN: ['B0014DZ4VW']
目标原始 ASIN: B0014DZ4VW
历史内部 item-id: [939]
目标内部 item-id: 939
历史标题: ['360-piece Solderless Electrical Terminal Assortment']
目标标题: 360-piece Solderless Electrical Terminal Assortment
目标时间: 2013-10-01T00:00:00+00:00

样本 3
原始用户 ID: A2KAOIY8M4D044
历史原始 ASIN: ['B001769ISQ']
目标原始 ASIN: B001769ISQ
历史内部 item-id: [1338]
目标内部 item-id: 1338
历史标题: ['American Terminal E-FMY250N-100 12/10-Gauge Economy Nylon Fully-Insulated Male Quick Disconnects']
目标标题: American Terminal E-FMY25

## 5. 在临时目录中调用切分与写文件函数

In [11]:
# 临时目录会在代码块结束后自动删除。
with TemporaryDirectory() as temp_dir:
    temp_args = SimpleNamespace(
        dataset=DATASET,
        output_path=temp_dir,
    )

    train_interactions, valid_interactions, test_interactions = (
        prep.convert_to_atomic_files_json2csv_style(
            temp_args,
            interaction_list,
            user2index,
        )
    )

    temp_output_dir = Path(temp_dir) / DATASET
    train_file = temp_output_dir / f"{DATASET}.train.inter"

    print("\n=== convert_to_atomic_files_json2csv_style 输出 ===")
    print(f"train 样本数: {len(train_interactions):,}")
    print(f"valid 样本数: {len(valid_interactions):,}")
    print(f"test 样本数: {len(test_interactions):,}")
    print("临时 train.inter 的前 3 行:")

    with train_file.open("r", encoding="utf-8") as file:
        for _ in range(SAMPLE_SIZE + 1):
            print(file.readline().rstrip())

Convert dataset: 
 Dataset:  Industrial_and_Scientific
Train interactions: 29458
Valid interactions: 3682
Test interactions: 3683

=== convert_to_atomic_files_json2csv_style 输出 ===
train 样本数: 29,458
valid 样本数: 3,682
test 样本数: 3,683
临时 train.inter 的前 3 行:
user_id:token	item_id_list:token_seq	item_id:token
3545	245	245
4150	939	939
3172	1338	1338


## 6. 构造评论文本字典

In [12]:
review_data = prep.load_review_data_amazon18_style(
    filtered_reviews,
    user2index,
    item2index,
)

print("\n=== load_review_data_amazon18_style 输出 ===")
print(f"review_data 条目数: {len(review_data):,}")
for key, value in list(review_data.items())[:SAMPLE_SIZE]:
    print(
        {
            "key": key,
            "review_preview": value["review"][:120],
            "summary": value["summary"],
        }
    )

# key(uid, iid, unix_timestramp)：内部用户id，内部商品id，交互时间

Load reviews: 100%|██████████| 43123/43123 [00:01<00:00, 26193.53it/s]


=== load_review_data_amazon18_style 输出 ===
review_data 条目数: 40,468
{'key': '(0, 5, 1511740800)', 'review_preview': 'This worked really well for what I used it for. So for my purposes it is getting full marks. This is an all around great', 'summary': "Couldn't have been happier with it's performance"}
{'key': '(1, 10, 1515801600)', 'review_preview': 'As advertised', 'summary': 'As advertised'}
{'key': '(2, 10, 1507334400)', 'review_preview': 'seems like a pretty good value as opposed to buying it at the big box stores by the sheet.', 'summary': 'seems like a pretty good value as opposed to buying it ...'}


## 7. 构造最终商品 metadata

In [13]:
item_features = prep.create_item_features_amazon18_style(
    metadata,
    item2index,
    id_title,
)

print("\n=== create_item_features_amazon18_style 输出 ===")
print(f"item_features 商品数: {len(item_features):,}")
for item_id, features in list(item_features.items())[:SAMPLE_SIZE]:
    print(
        {
            "item_id": item_id,
            "title": features["title"],
            "description_preview": features["description"][:160],
            "brand": features["brand"],
            "categories": features["categories"],
        }
    )


=== create_item_features_amazon18_style 输出 ===
item_features 商品数: 3,106
{'item_id': 0, 'title': 'SUPCO SPP6 Relay/Capacitor Hard Start Kit with 500% Increase Starting Torque', 'description_preview': "['Relay/CAPACITOR hard start kit 500% incr starting torque,increased compressor starting torque. Designed for use on permanent split CAPACITOR single phase a/c ", 'brand': 'Sealed Unit Parts Co., Inc.', 'categories': ''}
{'item_id': 1, 'title': 'Stanley TRA708T Sharpshooter 1/2-Inch Leg Length Staples, Steel (1000 Count)', 'description_preview': "['Heavy-duty staples are ideal for jobs such as insulation, carpet underlaying and roofing felt. Staple Type: Staple Gun; For Use With: Stanley Bostitch SharpSh", 'brand': 'Stanley', 'categories': ''}
{'item_id': 2, 'title': 'Kreg SML-C125-500 1-1/4-Inch #8 Coarse Pocket Hole Screws with Washer-Head, 500-Pack', 'description_preview': "['Kreg SML-C125-500 1-1/4-Inch #8 Coarse Pocket Hole Screws with Washer-Head, 500-Pack', 'The Kreg 1-1/4-Inch #8 

## 8. 预处理输出校验

运行完整预处理后执行下方单元；它只读取结果并打印基本统计和少量样本。

In [ ]:
# 此单元不会修改预处理结果，只读取文件进行校验。
import json

output_dir = OUTPUT_DIR / DATASET
split_names = ('train', 'valid', 'test')
required_files = [
    *(output_dir / f'{DATASET}.{split}.inter' for split in split_names),
    output_dir / f'{DATASET}.item.json',
    output_dir / f'{DATASET}.review.json',
    output_dir / f'{DATASET}.inter.json',
    output_dir / f'{DATASET}.user2id',
    output_dir / f'{DATASET}.item2id',
]
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError('缺少预处理输出：\n' + '\n'.join(str(path) for path in missing_files))

print(f'=== 输出目录：{output_dir} ===')

for split in split_names:
    split_file = output_dir / f'{DATASET}.{split}.inter'
    with split_file.open('r', encoding='utf-8') as file:
        header = file.readline().rstrip()
        sample_lines = [file.readline().rstrip() for _ in range(3)]
        row_count = 3 + sum(1 for _ in file)

    print(f'\n{split}.inter：{row_count:,} 条样本')
    print('表头：', header)
    print('前 3 行：')
    for line in sample_lines:
        print('  ', line)

with (output_dir / f'{DATASET}.item.json').open('r', encoding='utf-8') as file:
    item_data = json.load(file)
with (output_dir / f'{DATASET}.review.json').open('r', encoding='utf-8') as file:
    review_data = json.load(file)
with (output_dir / f'{DATASET}.inter.json').open('r', encoding='utf-8') as file:
    user_sequences = json.load(file)

print('\n=== JSON 文件统计 ===')
print(f'item.json 商品数：{len(item_data):,}')
print(f'review.json 评论条目数：{len(review_data):,}')
print(f'inter.json 用户数：{len(user_sequences):,}')
item_fields = sorted({field for item in item_data.values() for field in item})
nonempty_titles = sum(bool(str(item.get('title', '')).strip()) for item in item_data.values())
nonempty_descriptions = sum(bool(str(item.get('description', '')).strip()) for item in item_data.values())
sequence_lengths = [len(sequence) for sequence in user_sequences.values()]
print('item.json 字段：', item_fields)
print(f'非空 title：{nonempty_titles:,}/{len(item_data):,}')
print(f'非空 description：{nonempty_descriptions:,}/{len(item_data):,}')
print(f'用户交互序列长度：最小={min(sequence_lengths)}，平均={sum(sequence_lengths) / len(sequence_lengths):.2f}，最大={max(sequence_lengths)}')

sample_item_id = next(iter(item_data))
sample_review_key = next(iter(review_data))
sample_user_id = next(iter(user_sequences))

print('\nitem.json 样本：')
print({
    'item_id': sample_item_id,
    'title': item_data[sample_item_id].get('title', ''),
    'description_preview': str(item_data[sample_item_id].get('description', ''))[:160],
    'brand': item_data[sample_item_id].get('brand', ''),
})

print('review.json 样本：')
print({
    'key': sample_review_key,
    'review_preview': review_data[sample_review_key].get('review', '')[:160],
    'summary': review_data[sample_review_key].get('summary', ''),
})

print('inter.json 样本：')
print({
    'user_id': sample_user_id,
    'sequence_length': len(user_sequences[sample_user_id]),
    'first_item_ids': user_sequences[sample_user_id][:10],
})

for mapping_name in ('user2id', 'item2id'):
    mapping_file = output_dir / f'{DATASET}.{mapping_name}'
    with mapping_file.open('r', encoding='utf-8') as file:
        sample_mapping = [file.readline().rstrip() for _ in range(3)]
        mapping_count = 3 + sum(1 for _ in file)
    print(f'\n{mapping_name}：{mapping_count:,} 条映射；前 3 行：')
    for line in sample_mapping:
        print('  ', line)

split_total = sum(
    sum(1 for _ in (output_dir / f'{DATASET}.{split}.inter').open('r', encoding='utf-8')) - 1
    for split in split_names
)
print(f'\n三份 split 合计样本数：{split_total:,}')